<a id="unit-u.39d19df4a10540f89ab3cdcf3208ac06"></a>

<div>
  <div>10-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>Сығуды салыстыру және Partition өмірлік циклін тексеру</div>
  <p>Бақыланатын кіріспен кесте деңгейіндегі сығуды салыстырыңыз, толық сағаттық Partition жылжымалы терезесін қолдаңыз, терезеден тыс жазуды бақылаңыз және қолмен жойылған бір Partition қалпына келтіріңіз.</p>
  <span>Сығу · Range Partition · Dynamic Partitioning · Сақтау мерзімі · Recycle bin · RECOVER</span>
</div>

**10-модуль: Apache Doris жүйесінде сақтау шығындары мен деректердің өмірлік циклін басқару** · [Сабақ](course.md) · [Тест](quiz10_data_lifecycle_and_recovery.ipynb)

Шамамен 45 минут және қосымша нақты сағат бөлігіне шамамен бір сағатқа дейін жоспарлаңыз.

Сауда оқиға кестесі өсіп жатыр. Топ әр оқиғаға сақталатын байтты азайтып, шектелген онлайн уақыт терезесін ұстап, кездейсоқ Partition жоюынан қалпына келуі тиіс. Төрт қадаммен жұмыс істеңіз:

1. Бір құрылымды оқиға журналы кірісінде сығу таңдауларын салыстыру.
2. Ағымдағы сағат айналасында жылжымалы Dynamic Partition терезесін жасау.
3. Терезені бір сағат алға жылжытып, жоспарлағыш мерзімі өткен Partition бөлігін онлайн кестеден жоюын бақылау.
4. Бөлек қолмен Partition жойып, қалпына келтіру.

**Ұяшықтарды ретімен орындаңыз; Run All жұмыс істейді.** 3-бөлім сағат күтудің орнына терезені жылжытады, сондықтан Run All бірнеше минутта аяқталады. 3-бөлімнің соңындағы қосымша бөлік сол мерзім өтуді нақты сағатпен көрсетеді; келесі толық сағаттан кейін оралуыңыз қажет. Әр баптау бөлімі тек өз нысандарын бастапқы күйге келтіреді: 1-бөлімнің әр жүктеу ұяшығы өз сығу кестесін, 2-бөлім `m10_retained_events`, 4-бөлім `m10_recovery_events` қалпына келтіреді. Жұмыс `events_modelled` оқиды, бірақ ешқашан өзгертпейді. 1–8-модульдердің бір түйінді `doris-rta` ортасын қолданады және AWS S3, HDFS не қашықтағы сақтауға қатынаспайды.

<a id="unit-u.31ce38d9d0e943e79f3976dae191b13e"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты бастапқы баптау

Келесі ұяшықты 1-бөлімге дейін және Jupyter ядросын қайта бастағаннан кейін орындаңыз. Ол курс түбірін тауып, ортақ `DorisLab` көмекшісін жүктейді. Docker бастамайды және Doris деректерін өзгертпейді.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab
from doris_course.ui import card

lab = DorisLab(lab_dir=COURSE_ROOT);

<a id="unit-u.232b21667ac34ebe976c3095140bdfc9"></a>

<a id="1-compare-compression-choices-on-structured-event-logs"></a>

## 1. Құрылымды оқиға журналдарында сығу таңдауларын салыстыру

**Қажеттілік:** сауда аналитикасы тобы көп оқиға журнал хабарын сақтайды. Диск орны маңызды, бірақ жиі оқу Page ашуға CPU да жұмсайды. Қай кесте деңгейіндегі сығу баптауы әрі қарай сынауға лайық?

Келесі ұяшық 9-модуль кластері әлі жұмыс істесе алдымен тоқтайды: бір түйінді ортамен бірге курс минимумынан көп жад қажет етеді. Кейін бар жергілікті Doris 4.1.3 контейнерін бастайды не қайта пайдаланады, `doris_course` дерекқорын таңдайды және 4-зертханалық жұмыстың `events_modelled` кестесінің толықтығын тексереді. `events_modelled` өзгертпей бір күнін оқыңыз. Әр көшірілген жол детерминді синтетикалық `event_context` алады: `event_id` толық хабарды бірегей етеді, ал JSON өріс атаулары мен сипаттама мәтіні қайталанады. Бұл — бар бастапқы өріс не өлшенген өндірістік жүктеме емес, бақыланатын оқу кірісі.

Кіріс аралығы — **[2020-03-01 00:00:00, 2020-03-02 00:00:00)**. Бірінші шекара кіреді, екіншісі кірмейді. Жұмыс кестелерін қайта құрмас бұрын тек белгілі жол санын тексеріңіз.

In [ ]:
lab.check_module9_stopped()
lab.start_container()
lab.connect()
lab.execute("USE doris_course")
lab.check_prerequisites(10)

lab.sql("""
SELECT COUNT(*) AS event_count
FROM events_modelled
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
""", title="Compression input: one complete day");

<a id="unit-u.e53cb368e84249f292f41fb6f9123593"></a>

**Күтілетін нәтиже:** жасыл белгі `events_modelled` ішінде 10,158,080 жол барын растайды және `event_count` — **75,259**. Тексерулер дереккөзді оқиды, бірақ өзгертпейді. 9-модуль кластері жұмыс істеп тұрса, ұяшық ең алдымен тоқтап, оны тоқтататын пәрменді көрсетеді.

<a id="build-three-copies-of-the-same-input"></a>

### Сол кірістің үш көшірмесін құру

Келесі үш ұяшықтың әрқайсысы тек өзінің `m10_*` кестесін жояды, жасайды және жүктейді. Сәтсіз болса толық ұяшықты қайта орындаңыз; рет қайталанатын, бірақ бір атомарлы транзакция емес. Кестелердің бағандары, Duplicate Key реті, Range Partition, төрт Bucket және әр Tablet-ке бір репликасы бірдей. Тек `compression` кесте қасиеті өзгеше: `NO_COMPRESSION` (Page сығуынсыз кодталған деректер), `LZ4F` немесе жинақ әдепкісі `ZSTD`. `NO_COMPRESSION` — сыналған Doris 4.1.3 жинағы қабылдайтын жазылу.

Әр `INSERT ... SELECT` сол оқиға жолдарын көшіріп, **сол** құрылымды хабарды жасайды. Тұтас хабар `event_id` арқылы өзгереді, бірақ кілттері мен мәтінінің көбі өзгермейді. Кіріс пен орналасуды тең ұстау көрсетілген кесте өлшемдеріне сығу таңдауы туралы нақты сұраққа жауап беруге мүмкіндік береді.

In [ ]:
lab.execute("DROP TABLE IF EXISTS m10_events_none FORCE")
lab.execute("""
CREATE TABLE m10_events_none (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL,
    event_context STRING NOT NULL
)
DUPLICATE KEY(event_time, event_id, user_id)
PARTITION BY RANGE(event_time) (
    PARTITION p20200301 VALUES [('2020-03-01'), ('2020-03-02'))
)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES (
    "replication_num" = "1",
    "compression" = "NO_COMPRESSION"
)
""")
lab.insert("""
INSERT INTO m10_events_none
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue,
       CONCAT(
           '{"source":"commerce-web","schema":"event-v1",',
           '"pipeline":"real-time-analytics",',
           '"message":"A customer event entered the commerce analytics pipeline.",',
           '"event_id":"', CAST(event_id AS STRING), '",',
           '"event_type":"', event_type, '",',
           '"region":"', region, '",',
           '"product_id":"', CAST(product_id AS STRING), '"}'
       ) AS event_context
FROM events_modelled
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
""", title="Load the NO_COMPRESSION table", expected_rows=75_259);

In [ ]:
lab.execute("DROP TABLE IF EXISTS m10_events_lz4f FORCE")
lab.execute("""
CREATE TABLE m10_events_lz4f (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL,
    event_context STRING NOT NULL
)
DUPLICATE KEY(event_time, event_id, user_id)
PARTITION BY RANGE(event_time) (
    PARTITION p20200301 VALUES [('2020-03-01'), ('2020-03-02'))
)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES (
    "replication_num" = "1",
    "compression" = "LZ4F"
)
""")
lab.insert("""
INSERT INTO m10_events_lz4f
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue,
       CONCAT(
           '{"source":"commerce-web","schema":"event-v1",',
           '"pipeline":"real-time-analytics",',
           '"message":"A customer event entered the commerce analytics pipeline.",',
           '"event_id":"', CAST(event_id AS STRING), '",',
           '"event_type":"', event_type, '",',
           '"region":"', region, '",',
           '"product_id":"', CAST(product_id AS STRING), '"}'
       ) AS event_context
FROM events_modelled
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
""", title="Load the LZ4F table", expected_rows=75_259);

In [ ]:
lab.execute("DROP TABLE IF EXISTS m10_events_zstd FORCE")
lab.execute("""
CREATE TABLE m10_events_zstd (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL,
    event_context STRING NOT NULL
)
DUPLICATE KEY(event_time, event_id, user_id)
PARTITION BY RANGE(event_time) (
    PARTITION p20200301 VALUES [('2020-03-01'), ('2020-03-02'))
)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES (
    "replication_num" = "1",
    "compression" = "ZSTD"
)
""")
lab.insert("""
INSERT INTO m10_events_zstd
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue,
       CONCAT(
           '{"source":"commerce-web","schema":"event-v1",',
           '"pipeline":"real-time-analytics",',
           '"message":"A customer event entered the commerce analytics pipeline.",',
           '"event_id":"', CAST(event_id AS STRING), '",',
           '"event_type":"', event_type, '",',
           '"region":"', region, '",',
           '"product_id":"', CAST(product_id AS STRING), '"}'
       ) AS event_context
FROM events_modelled
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
""", title="Load the ZSTD table", expected_rows=75_259);

<a id="unit-u.6f0c677e5dbb4e8ab574a7fd45b54b36"></a>

<a id="inspect-the-values-then-compare-storage-reports"></a>

### Мәндерді тексеріп, содан кейін сақтау есептерін салыстыру

`event_type` сияқты бірегей мәні аз баған **баған кодтауынан** кейін ықшам болуы мүмкін. Бұл Page сығуынан кесте өлшемі көп азаятынын кепілдемейді. Керісінше, толық `event_context` бірегей, бірақ қайталанатын JSON кілттері мен хабар мәтіні ортақ **байт үлгілерін** береді. Кесте өлшемі салыстыруын оқымас бұрын екі пішінді де тексеріңіз. Үш таңдаудың пайдалану мақсаттары бөлек:

| Таңдау | Doris Page қабатында не істейді | Негізгі таңдау салдары | Таңдаудың әдеттегі себебі |
|---|---|---|---|
| `NO_COMPRESSION` | Баған кодтауын сақтап, Page сығуын өткізеді | Page сығу/ашу жұмысын болдырмайды, бірақ әдетте көп байт оқып, сақтайды | Мазмұн сығылған немесе CPU шығыны сақтаудан маңызды |
| `LZ4F` | Әр кодталған Page LZ4 Frame пішімімен сығады | Жылдам, орташа сығу қатынасы | Doris жылдам таңдау нұсқаулығы сақтау шығыны мен сұрау өнімділігі теңгерімі үшін ZSTD жанында тізеді |
| `ZSTD` | Әр кодталған Page Zstandard арқылы сығады | Жоғары сығу қатынасы; ашу жылдам қалады | Сол теңгерім; Doris әдепкі ұсынысы және осы жинақ әдепкісі |

Бұл қасиеттер Doris Data Compression бетінен алынған; осы шағын жұмыс өлшеген қорытындылар емес. Эксперимент мән сандарын, тең бекітілген жол сандарын және Doris ағымдағы сақтау есебін тікелей бақылайды. CPU тұтынуын немесе өндірістік сұрау кідірісін өлшемейді.

Барлық Tablet өлшемі мен жол саны FE түйініне жеткенге дейін бекітілген жолдар сұрауға қолжетімді болады. Көмекші алдымен әр кестені `SELECT COUNT(*)` арқылы тексереді, кейін `SHOW DATA` 75,259 жол және нөлден үлкен өлшем көрсеткенше күтеді. Күту 180 секундпен шектелген және шексіз жүрмей, соңғы метадеректермен сәтсіз аяқталады.

Соңғы кесте төрт дәлел бөлігін біріктіреді:

- әдепкі емес сығу қасиетіне `SHOW CREATE TABLE`;
- Doris қайтарған DDL ішінен әдепкіні алып тастағанда FE `default_compression_type`;
- ағымдағы физикалық есепке `SHOW DATA`, `SHOW PARTITIONS` және `SHOW TABLETS`;
- сол метадерек көшірмесінен есептелген `m10_events_none` салыстырмалы үнем пайызы.

<a id="unit-u.f1a19dc191b94e5890d9902b4c762086"></a>

<a id="see-two-event-types-and-a-shared-message-prefix"></a>

### Екі оқиға түрі мен ортақ хабар префиксін көру

Бастапқы күнде тек екі `event_type` мәні бар. Төмендегі сұрау әр мән мен жол санын көрсетеді. **Бірегей толық** `event_context` жолдарын санап, әр түрдің бір мысалынан қысқа префикс көрсетеді. Екі префикстен сол `source`, `schema` және `pipeline` кілттерін іздеңіз. `COUNT(DISTINCT ...)` мәндерді санайды; жолдарды алып тастамайды, физикалық баған өлшемдерін ашпайды және Doris қай кодтауды таңдағанын дәлелдемейді.


In [ ]:
value_patterns = lab.sql("""
SELECT event_type, COUNT(*) AS event_count,
       COUNT(DISTINCT event_context) AS distinct_contexts,
       SUBSTRING(MIN(event_context), 1, 92) AS shared_prefix
FROM m10_events_none
GROUP BY event_type
ORDER BY event_type
""", title="Two event types, unique messages, shared bytes")
assert value_patterns["event_type"].tolist() == ["cart", "purchase"]
assert value_patterns["event_count"].astype(int).tolist() == [56_961, 18_298]
assert (value_patterns["event_count"] == value_patterns["distinct_contexts"]).all()
assert value_patterns["shared_prefix"].nunique() == 1


<a id="unit-u.3f9cd52a698e4125b4b20d158bf76a3d"></a>

**Күтілетін нәтиже:** екі жол — `cart` (**56,961**) және `purchase` (**18,298**). Әр түр үшін `distinct_contexts = event_count`: ешбір толық жасалған хабар қайталанбайды. Толық жолдары өзгеше болса да, көрсетілген префикстердің JSON кілттері мен сипаттама құрылымы ортақ. Екі оқиға түрін баған кодтауымен көрсету арзан болуы мүмкін, ал бірегей хабарлардағы қайталанатын ішкі жолдар Page сығуына бөлек мүмкіндік береді. Ешбір нәтиже әр бағанның физикалық сығу қатынасын көрсетпейді.

Содан кейін сол кірісті қамтитын үш кестені салыстырыңыз. `SHOW DATA` әр **тұтас кестені** көрсетеді, сондықтан нәтиже `event_type` не `event_context` жеке қанша байт үнемдегенін оқшаулай алмайды.


In [ ]:
compression_tables = [
    "m10_events_none",
    "m10_events_lz4f",
    "m10_events_zstd",
]
lab.wait_for_storage_reports(
    compression_tables,
    expected_rows=75_259,
    timeout_seconds=180,
)
storage_comparison = lab.compression_storage_summary(
    compression_tables,
    baseline_table="m10_events_none",
);

<a id="unit-u.8dd5becc236b478aa680b22e771bd683"></a>

**Нәтижені қалай түсіндіру керек:**

- `effective compression` салыстыру нақты `NO_COMPRESSION`, `LZ4F` және `ZSTD` қолданатынын растайды.
- Үш жол да 75,259 логикалық жол, бір Partition, төрт Tablet және әр Tablet-ке бір реплика көрсетеді. Бірдей `INSERT ... SELECT` кірісімен бірге осы тең мәндер бақыланатын салыстыруды анықтайды.
- `reported stored size` — Doris осы үлгіге қазір көрсететін көлем. `reported space saved vs baseline` сол мәндерді `NO_COMPRESSION` салыстыруына айналдырады; оң пайыз — көрсетілген байттар аз.
- `ZSTD` `resolved from = FE default` деуі мүмкін. Бұл жинақта көрінетін `CREATE TABLE` `ZSTD` айқын сұраса да, `SHOW CREATE TABLE` ағымдағы әдепкіге тең қасиетті түсіреді.

Нәтижені екі жолдық мән үлгісі тексеруімен салыстырыңыз: жасалған контексте толық жол көшірмелері жоқ, бірақ қайталанатын JSON кілттері мен хабар мәтіні Page сығуына қайталанатын байт береді. `event_type` баған кодтауынан кейін ықшам болуы мүмкін; тұтас кесте айырмасын тек нақты сақтау өлшемі анықтайды. Файл тақырыптары, индекстер, Segment метадеректері, төрт Tablet және дөңгелектеу кесте жиынтықтарына әлі әсер етеді. Өлшенген айырмашылық осы құрама кесте мен орналасуға тиесілі; `event_context` оқшауламайды және әр баған сығу қатынасын дәлелдемейді. Тексерілген жергілікті 4.1.3 жинағында ортақ мәтінді кіріс `LZ4F` және `ZSTD` екеуіне де айқын кесте өлшемі азаюын берді: көрсетілген өлшемдер `NO_COMPRESSION` үшін шамамен 18.0 MB, `LZ4F` үшін 2.4 MB (86.6% аз), `ZSTD` үшін 1.9 MB (89.7% аз). Нақты мәндеріңіз өзгеше болуы мүмкін.

**Дәлел бойынша таңдаңыз:** `NO_COMPRESSION` — бастапқы жолдардың шикі сығылмаған өлшемі емес, кодталған кесте бастапқы нұсқасы. Көрсетілген пайыз тек Doris хабарлаған кесте өлшемдерінің айырмасы. `LZ4F` пен `ZSTD` екеуін де кодталған `NO_COMPRESSION` нұсқасымен салыстырыңыз. Үлкен азаю сауда оқиғаларының әмбебап қатынасын емес, осы ортақ мәтін жүктемесінің әсерін көрсетеді. Екеуінің кішісі — осы үлгінің сақтау нәтижесі; CPU шығыны немесе сұрау кідірісі бойынша жеңетінін дәлелдемейді. Оны мәлімдемес бұрын өкілдік өндірістік көлемді жүктеу және қайталанатын жүктеме өлшемдері қажет. Мұнда екі алгоритм де бастапқыдан едәуір үнемдейді, бірақ бір-бірінен аз ерекшеленеді, шамамен 2.4 MB және 1.9 MB: осы дәлелмен әдепкі `ZSTD` сақтаңыз. Екеуі бастапқыдан аз ерекшеленсе де, пайдалы нәтиже болар еді: **басқа алгоритм бар екен деп сығу қасиетін өзгертпеңіз.**

<a id="unit-u.edf4feb15abc47fdbc44ccd830dcd50f"></a>

<a id="2-create-an-hourly-dynamic-partition-window"></a>

## 2. Сағаттық Dynamic Partition терезесін жасау

Сығу сақталған байттарды азайтады; сақтау мерзімі **қай толық уақыт ауқымдары онлайн қалатынын** шешеді. Осы не келесі бөлімнің ешбір ұяшығы `DROP PARTITION` бермейді: мерзімі өткен ауқымды Dynamic Partition жоспарлағышы жояды.

Кесте `event_time` **DATETIME** Partition кілтін және Doris сеанс сағаты мен жоспарлағышына `Asia/Shanghai` пайдаланады. `time_unit = HOUR`, `start = -3` және `end = 1` алдыңғы үш толық сағатты, ағымдағы және бір болашақ сағатты жасайды. `create_history_partition = true` ескі сағаттарды бірден қолжетімді етеді. Терезеде әр жолға жылжымалы үш сағаттық таймер емес, **бес сағаттық Partition** бар.

Келесі ұяшық тек `m10_retained_events` қайта құрып, бес алғашқы Partition күтеді. Жаңа бақылауды бастауға бір рет орындаңыз. Қайта орындау дәлелді жойып, қайта жасайды.

In [ ]:
lab.execute("SET time_zone = 'Asia/Shanghai'")
lab.execute("DROP TABLE IF EXISTS m10_retained_events FORCE")
lab.execute("""
CREATE TABLE m10_retained_events (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    note VARCHAR(64) NOT NULL
)
DUPLICATE KEY(event_time, event_id)
PARTITION BY RANGE(event_time) ()
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES (
    "replication_num" = "1",
    "dynamic_partition.enable" = "true",
    "dynamic_partition.time_unit" = "HOUR",
    "dynamic_partition.start" = "-3",
    "dynamic_partition.end" = "1",
    "dynamic_partition.prefix" = "p",
    "dynamic_partition.buckets" = "1",
    "dynamic_partition.create_history_partition" = "true",
    "dynamic_partition.time_zone" = "Asia/Shanghai"
)
""")
lab.wait_for_partition_count(
    "m10_retained_events", expected_count=5, timeout_seconds=60
);

<a id="unit-u.3165ceccc74b4c54a5a7da30b9ba16d5"></a>

Үш сағат бұрынғы оқиғаны **ең ескі сақталған сағаттық Partition** ішіне енгізіңіз. Терезе жылжығанда не қалатынын көру үшін екі жаңарақ оқиға қосыңыз. Ең ескі тұтас Partition қашан жойылатынын оқиға деңгейіндегі таймер емес, жоспарлағыш шешеді.

Белгі сұрауы нақты `event_time` және `eligible_at` шығарады: нақты сағатта Partition `start = -3` сыртында қалатын келесі толық сағат шекарасы. Бұл уақыт 3-бөлімнің тек қосымша нақты сағат бөлігіне қажет. Төмендегі `SHOW PARTITIONS` нақты жартылай ашық сағат ауқымдарын және жолдарын көрсетеді. 16:xx кезінде баптасаңыз, ең ескі Partition 13:00 басталады, ал ең ерте жою жарамдылығы Asia/Shanghai 17:00 болады. Әдепкі FE жоспарлағышы кейінірек әрекет етуі мүмкін.

In [ ]:
lab.insert("""
INSERT INTO m10_retained_events VALUES
    (DATE_SUB(NOW(), INTERVAL 3 HOUR), 1001, 'oldest retained hour'),
    (DATE_SUB(NOW(), INTERVAL 2 HOUR), 1002, 'newer hour'),
    (NOW(),                             1003, 'current hour')
""", title="Insert three known hourly events", expected_rows=3)

lab.sql("""
SELECT event_time, event_id, note,
       DATE_ADD(DATE_TRUNC(event_time, 'hour'), INTERVAL 4 HOUR) AS eligible_at
FROM m10_retained_events
WHERE event_id = 1001
""", title="Oldest event and its next expiry boundary")
lab.dynamic_partition_status(
    "m10_retained_events", title="Scheduler after initial creation"
)
window_before_expiry = lab.partition_window_summary(
    "m10_retained_events", title="Hourly Partitions before the clock advances"
)


<a id="unit-u.c12cf27a53ae42a999a6774346c9a157"></a>

**Күтілетін нәтиже:** `1001` оқиғасы бар және `eligible_at` — `Asia/Shanghai` келесі толық сағат шекарасы. Бес Partition ауқымы үш сағат бұрын, екі сағат бұрын, бір сағат бұрын, осы сағат және келесі сағаттан басталады. Үшеуінде бір-бір оқиға бар; алдыңғы бір сағат және болашақ сағат бөлімдері бос. Нақты уақыт белгілері мен атаулар бастау уақытына байланысты.

`NOW() - INTERVAL 3 HOUR` оқиғасының жасы үш сағат болса да, толық сағаты **келесі сағат шекарасына** және одан кейінгі жоспарлағыш тексеруіне дейін онлайн қалады. Dynamic Partitioning әр жолды жасы үш сағат болған сәтте емес, тұтас сағатты жояды.


<a id="unit-u.cfc8dd2370c54f12a6414602f21b5af3"></a>

<a id="3-move-the-window-forward-and-observe-automatic-expiry"></a>

## 3. Терезені алға жылжытып, автоматты мерзім өтуді бақылау

Нақты сағатта `1001` оқиғасының сағаты `eligible_at` кезінде терезеден шығып, жоспарлағыш келесі тексеруде жояды. Әдепкі 600 секундтық тексеруде бұл баптаудан кейін 70 минутқа дейін созылуы мүмкін. Күтудің орнына келесі сағат жасайтын өзгерісті жасаңыз:

1. `dynamic_partition.start` `-3` мәнін `-2` етіп тарылтыңыз. Терезе енді үштің орнына алдыңғы екі сағатты сақтайды, дәл `-3` кезінде бір сағаттан кейінгідей, сондықтан `1001` сағаты сыртында қалады.
2. FE `dynamic_partition_check_interval_seconds` мәнін аз уақытқа 10 етіп қысқартыңыз. Жоспарлағыш аралық өзгерісін бірнеше секундта байқап, `start` мәнінен ескі әр Partition жоятын тексеру жасайды.
3. Аралықты бұрынғы мәніне қайтарыңыз. `ADMIN SET FRONTEND CONFIG` баптауды бір кестеге емес, тұтас FE үшін өзгертеді.

Жұмыс әлі `DROP PARTITION` бермейді: ережені жоспарлағыш қолданады. Қысқа жол шекараның өздігінен келуін емес, жоюды көрсетеді; бөлім соңындағы қосымша бөлік нақты сағатты көрсетеді. Өндірісте осы өзгерістерді ерте дерек шығару үшін қолданбаңыз.

Ұяшық алдымен `1001` оқиғасы бар Partition табады. Оқиға жоқ болса, мысалы нақты сағаттан кейін оралсаңыз, ештеңе өзгертпейді.

In [ ]:
expired_partition = lab.partition_holding_event("m10_retained_events", event_id=1001)

if expired_partition is not None:
    lab.execute("""
    ALTER TABLE m10_retained_events SET ("dynamic_partition.start" = "-2")
    """)

    interval = lab.sql(
        "SHOW FRONTEND CONFIG LIKE 'dynamic_partition_check_interval_seconds'",
        title="Scheduler check interval before the shortcut",
        columns=["Key", "Value"],
    )
    original_interval = interval.iloc[0]["Value"]
    lab.execute("ADMIN SET FRONTEND CONFIG ('dynamic_partition_check_interval_seconds' = '10')")
    try:
        lab.wait_for_partition_removal(
            "m10_retained_events", expired_partition, timeout_seconds=120
        )
    finally:
        lab.execute(
            f"ADMIN SET FRONTEND CONFIG ('dynamic_partition_check_interval_seconds' = '{original_interval}')"
        );

<a id="unit-u.70968a8955ec4d50ae300ca138ffa046"></a>

**Күтілетін нәтиже:** бірінші карточка бес бөлімнің ең ескісі, `1001` оқиғасы бар Partition атайды. `Scheduler check interval before the shortcut` `600` көрсетеді. Кейін таймер жоспарлағыш сол Partition жойғанын, әдетте аралық өзгергеннен кейін 10 секунд ішінде көрсетеді және ұяшық соңында аралық `600` мәніне қайтады. Partition жойған осы ұяшық емес, жоспарлағыш.

<a id="check-the-oldest-event-and-current-window"></a>

### Ең ескі оқиға мен ағымдағы терезені тексеру

`1001` оқиғасына **сол** сұрауды орындап, ағымдағы Partition ауқымдарын тексеріңіз. Бұл тек оқу тексеруі қысқа жол мен нақты сағат жолына ортақ.

In [ ]:
after_expiry = lab.sql("""
SELECT event_time, event_id, note,
       DATE_ADD(DATE_TRUNC(event_time, 'hour'), INTERVAL 4 HOUR) AS eligible_at
FROM m10_retained_events
WHERE event_id = 1001
""", title="After hourly rollover: oldest event no longer online")
assert after_expiry.empty

lab.dynamic_partition_status(
    "m10_retained_events", title="Scheduler after automatic removal"
)
window_after_expiry = lab.partition_window_summary(
    "m10_retained_events", title="Hourly Partitions after automatic removal"
)


<a id="unit-u.90355b589cff41198cfc7d3779f9f6b2"></a>

**Күтілетін нәтиже:** қайталанған `SELECT` `1001` оқиғасы үшін **0 жол** қайтарады; оның бастапқы сағаттық Partition бөлімі жоқ.

- **Қысқа жолдан** кейін (`start = -2`) терезеде **төрт** Partition бар: алдыңғы екі сағат, ағымдағы және келесі сағат.
- **Нақты сағат жолынан** кейін (`start = -3`) терезеде **бес** Partition бар: алдыңғы үш сағат, ағымдағы және келесі сағат. Ауқымдары 2-бөлімнен бері бір сағат алға жылжыған.

`1002` және `1003` оқиғалары сағаттары ағымдағы терезеде тұрғанда қалады. Қосымша сағат шекараларынан кейін оралу терезені тағы жылжыта алады. Күй карточкасы жоспарлағыштың дұрыстығын, ал өзгерген SQL жауабы мен Partition ауқымдары нақты онлайн деректерді көрсетеді.

Жоғалу әр Backend (BE) сақтау байтының жедел босатылуын емес, логикалық сақтау мерзімі шекарасын дәлелдейді. Recycle bin сақтау мерзімі мен фондық тазалау бөлек.

<a id="try-an-event-outside-the-current-window"></a>

### Ағымдағы терезеден тыс оқиғаны сынау

**Ағымдағы** сағат шекарасынан төрт сағат бұрынғы енгізу уақыты жоғарыдағы екі терезеден де тыс. Келесі ұяшық нысаналы жетіспейтін Partition қатесін талап етеді; ескі Partition қайта жасамайды.

In [ ]:
late_write_error = lab.expect_statement_error("""
INSERT INTO m10_retained_events VALUES
    (DATE_SUB(DATE_TRUNC(NOW(), 'hour'), INTERVAL 4 HOUR),
     1099, 'outside hourly window')
""", contains="no partition for this tuple",
     title="Late event rejected outside the hourly window")

retained_rows = lab.sql("""
SELECT event_time, event_id, note
FROM m10_retained_events
ORDER BY event_time, event_id
""", title="Rows that remain online")
assert 1001 not in retained_rows["event_id"].astype(int).tolist()


<a id="unit-u.d7e03d9b9abd4c5c82df36636615325c"></a>

**Күтілетін нәтиже:** Doris төрт сағат бұрынғы оқиға үшін `no partition for this tuple` көрсетеді. Онлайн кестеде `1002` және `1003` бар, ал `1001` және қабылданбаған `1099` жоқ.

Уақыты сақталған Partition ішінде болса, кеш оқиға әлі қабылдана алады. Терезеден тыс уақытқа бөлек бизнес шешімі қажет: қабылдамау, басқаға бағыттау немесе тарихи Partition айқын басқару. Dynamic Partitioning бұл шешімді автоматты жасамайды.

<a id="unit-u.34c6c05b2500405b90637a8dd1bccbf9"></a>

<a id="optional-watch-the-same-expiry-on-the-real-clock"></a>

### Қосымша: сол мерзім өтуді нақты сағатпен бақылау

Қысқа жол ережені өзгертті. Partition ешбір өзгеріссіз мерзімінен шығуын көру үшін сағатқа терезені жылжытуға мүмкіндік беріңіз:

1. 2-бөлімнің екі ұяшығын қайта орындаңыз. Бұл `start = -3` бар `m10_retained_events` қайта құрып, жаңа `eligible_at` шығарады.
2. Осы бөлімнің қысқа жол ұяшығын өткізіп жіберіңіз.
3. Төменде `WAIT_FOR_REAL_HOUR = True` орнатып, ұяшықты `eligible_at` кезінде не одан кейін орындаңыз. Арада ноутбукты жаба аласыз; контейнерді жұмыс істетіп қалдырыңыз. Kernel қайта басталса, алдымен **Initialize the Lab** орындаңыз.

Ұяшықты `eligible_at` дейін орындасаңыз, қашан оралу керектігін көрсетіп, тоқтайды. Шекарадан кейін тікелей таймер жоспарлағыштың келесі тексеруін ең көбі 15 минут күтеді; әдепкі тексеру 600 секунд сайын. Содан кейін баптауды не қысқа жолды қайта орындамай [Ең ескі оқиға мен ағымдағы терезені тексеру](#check-the-oldest-event-and-current-window) орындаңыз. Ол `1001` үшін 0 жол қайтарып, бұрынғыдан бір сағат кейінгі бес Partition көрсетеді.

In [ ]:
WAIT_FOR_REAL_HOUR = False  # set to True after rebuilding Section 2, at or after eligible_at

if WAIT_FOR_REAL_HOUR:
    lab.start_container()
    lab.connect()
    lab.execute("USE doris_course")
    lab.execute("SET time_zone = 'Asia/Shanghai'")
    lab.wait_for_hourly_partition_expiry(
        "m10_retained_events", event_id=1001,
        start_offset=-3, timeout_seconds=900,
    )
else:
    card("Set WAIT_FOR_REAL_HOUR = True to run this part.", "skip",
         "Optional: real-clock expiry skipped");

<a id="unit-u.cc1c95d6358c453cb0500ea0c002abbe"></a>

<a id="4-recover-a-manually-dropped-partition-without-a-pre-recorded-id"></a>

## 4. Алдын ала тіркелген ID жоқ қолмен жойылған Partition қалпына келтіру

Автоматты сақтау мерзімі мен кездейсоқ жоюдың пайдалану рәсімдері бөлек. Жоспарлағыш қалпына келтіру мысалын өзгертпес үшін бөлек қолмен Partition бөлінген кесте пайдаланыңыз.

`p_recover` белгілі екі жол, `p_keep` бір жол қамтиды. **`FORCE` болмаса**, `ALTER TABLE ... DROP PARTITION p_recover` Partition бөлігін онлайн кестеден алып, шектеулі уақытқа catalog recycle bin ішіне қояды, сондықтан `RECOVER PARTITION` қайтара алады. **`FORCE` болса**, Doris осы жолды айналып өтеді: бұл жою `RECOVER` арқылы кері қайтпайды. Эксперимент `FORCE` әдейі түсіріп, Partition ID тек жойылғаннан кейін табады.

Баптау ұяшығы `DROP TABLE IF EXISTS m10_recovery_events FORCE` тек жұмысқа тиесілі кестенің ескі көшірмесін бастапқы күйге келтіруге қолданады. Ол recycle bin арқылы қалпына келмейді және төмендегі қалпына келетін Partition жоюынан **бөлек**.

Әдепкіде Doris қалпына келетін catalog жазбаларын жойылғаннан кейін **86,400 секунд (24 сағат)** сақтайды; Frontend (FE) `catalog_trash_expire_second` баптауы осы кезеңді басқарады. Сыналған жергілікті Doris 4.1.3 ортасы қазір де `86400` көрсетеді. Мұны кепіл емес, шектеулі мүмкіндік деп қараңыз: әкімші баптауды өзгерте алады, жазба тазартылуы мүмкін және `FORCE` жоюы бұл жолға ешқашан кірмейді. `RECOVER` сынамас бұрын `SHOW CATALOG RECYCLE BIN` тексеріңіз.


In [ ]:
lab.execute("DROP TABLE IF EXISTS m10_recovery_events FORCE")
lab.execute("""
CREATE TABLE m10_recovery_events (
    event_date DATE NOT NULL,
    event_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL,
    note VARCHAR(64) NOT NULL
)
DUPLICATE KEY(event_date, event_id)
PARTITION BY RANGE(event_date) (
    PARTITION p_recover VALUES [('2020-03-01'), ('2020-03-02')),
    PARTITION p_keep    VALUES [('2020-03-02'), ('2020-03-03'))
)
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO m10_recovery_events VALUES
    ('2020-03-01', 2001, 19.95, 'recover me A'),
    ('2020-03-01', 2002, 29.95, 'recover me B'),
    ('2020-03-02', 2003,  9.95, 'keep me')
""", title="Insert three known recovery rows", expected_rows=3)

before_drop = lab.sql("""
SELECT event_date, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m10_recovery_events
GROUP BY event_date
ORDER BY event_date
""", title="Online answer before the accidental drop")
assert before_drop["event_count"].astype(int).tolist() == [2, 1];

<a id="unit-u.421c5b3e322a4fc3bcfd9f709b39e560"></a>

**Күтілетін нәтиже:** 1 наурызда жалпы **49.90** болатын екі жол; 2 наурызда жалпы **9.95** болатын бір жол бар.

Содан кейін `p_recover` бөлігін **`FORCE` қолданбай** жойып, recycle bin атауы бойынша сұраңыз. Метадеректеріне жойылған нысан түрі, кесте мен Partition ID және жою уақыты кіреді. Мұнда `FORCE` қосу осы жою үшін келесі `RECOVER PARTITION` мүмкіндігін жояр еді. Алдыңғы орындау `p_recover` атты басқа жазба қалдыруы мүмкін; қалай қалпына келтіруді таңдамас бұрын жазбаларды қараңыз.


In [ ]:
lab.execute("""
ALTER TABLE m10_recovery_events
DROP PARTITION p_recover
""")

after_drop = lab.sql("""
SELECT event_date, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m10_recovery_events
GROUP BY event_date
ORDER BY event_date
""", title="Online answer after dropping p_recover")
assert after_drop["event_count"].astype(int).tolist() == [1]
assert str(after_drop.iloc[0]["total_revenue"]) == "9.95"

recycle_entries = lab.sql(
    "SHOW CATALOG RECYCLE BIN WHERE NAME = 'p_recover'",
    title="Recoverable entries named p_recover",
    columns=("Type", "Name", "TableId", "PartitionId", "DropTime"),
)
assert (recycle_entries["Type"].str.lower() == "partition").any()


<a id="unit-u.9f6e16aa6798498e9540e09f60c503ce"></a>

**Күтілетін нәтиже:** тек 2 наурыз жолы сұрауға қолжетімді қалады. Recycle bin нәтижесінде `p_recover` атты `Partition` және жою уақыты бар. Жұмыс бұрын орындалса, сол атаудағы ескі жазбалар да болуы мүмкін. Бұл ID диагностикаға көрсетіледі; келесі пәрмен әдейі біреуін де бермейді.

Doris тек атаулы `RECOVER PARTITION` сол атаудың ең соңғы жойылған метадеректері деп түсінеді. Мұнда оқшауланған жұмыста жою жаңа болды, сондықтан таңдау әдейі. Нақты оқиғада `TableId`, `PartitionId` және `DropTime` тексеріп, ең жаңа жазба қажетіңіз болмауы мүмкін кезде дұрыс ID көрсетіңіз. Қалпына келу үшін жазба recycle bin ішінде жарамды қалуы тиіс.


In [ ]:
lab.execute("""
RECOVER PARTITION p_recover
FROM m10_recovery_events
""")

recovered = lab.sql("""
SELECT event_date, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m10_recovery_events
GROUP BY event_date
ORDER BY event_date
""", title="Online answer after RECOVER PARTITION")
assert recovered["event_count"].astype(int).tolist() == [2, 1]
assert [str(value) for value in recovered["total_revenue"]] == ["49.90", "9.95"]
card("The just-dropped Partition and its two known rows are online again.",
     "ok", "Partition recovery verified");

<a id="unit-u.9e3242874e134a0aa176b89c9b41362e"></a>

**Күтілетін нәтиже:** екі күндік жауап 1 наурызға екі жол және **49.90**, 2 наурызға бір жол және **9.95** қайтарады. Бұл осы орындауда атаумен қалпына келтіру көзделген логикалық деректерді қайтарғанын тексереді.

Шекара маңызды:

- **`FORCE` жоқ** қолмен Partition жою catalog recycle bin ішіне кіріп, жазбасы жарамды кезде қалпына келе алады;
- ID түсіру сол атаудың ең соңғы жойылған нысанын таңдайды, ол нақты оқиғада сіз көздеген нысан болмауы мүмкін;
- `DROP PARTITION ... FORCE` қалпына келетін recycle bin айналып өтеді, сондықтан `RECOVER PARTITION` оны кері қайтара алмайды;
- жол деңгейіндегі `DELETE`, автоматты сақтау мерзімі және тұтас жоғалған кластердің қалпына келу әрекеті бірдей деп болжанбауы тиіс;
- recycle bin тәуелсіз сақтық көшірме емес. Кластерлер арасындағы апаттан қалпына келуге сыналған сақтық көшірме не беделді дереккөзді қайта ойнату қажет.


<a id="unit-u.9c984b83b6d14881a97c2e594e8f5d66"></a>

<a id="5-practice-predict-a-daily-window-before-you-create-it"></a>

## 5. Жаттығу: күндік терезені жасамас бұрын болжау

Сабақтың өндірістік саясаты — күндік терезе. `DATETIME` Partition кілті және мына Dynamic Partition қасиеттері бар `m10_practice_window` жаттығу кестесін жасаңыз: `time_unit = DAY`, `start = -2`, `end = 2`, `prefix = p`, `create_history_partition = true`, `time_zone = Asia/Shanghai`.

Жасамас бұрын жазып алыңыз:

- Doris бірден қанша Partition жасайды және бүгінгі Asia/Shanghai күні үшін бірінші мен соңғысының атаулары;
- бірінші Partition төменгі және жоғарғы шекарасы;
- оқиға уақыты қазірден үш күн бұрын болатын енгізумен не болады.

Содан кейін кесте жасап, `lab.partition_window_summary` арқылы бөлімдерін көрсетіңіз, `lab.expect_statement_error` арқылы ескі оқиғаны сынап, кестені жойыңыз.

In [ ]:
# Your code here

<a id="unit-u.4b499f9373044e449d187b12bd172567"></a>

Келесі ұяшық — үлгі шешім; ол жиналған. Салыстыру үшін оны ашыңыз немесе орындаңыз.

In [ ]:
lab.execute("SET time_zone = 'Asia/Shanghai'")
lab.execute("DROP TABLE IF EXISTS m10_practice_window FORCE")
lab.execute("""
CREATE TABLE m10_practice_window (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL
)
DUPLICATE KEY(event_time, event_id)
PARTITION BY RANGE(event_time) ()
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES (
    "replication_num" = "1",
    "dynamic_partition.enable" = "true",
    "dynamic_partition.time_unit" = "DAY",
    "dynamic_partition.start" = "-2",
    "dynamic_partition.end" = "2",
    "dynamic_partition.prefix" = "p",
    "dynamic_partition.buckets" = "1",
    "dynamic_partition.create_history_partition" = "true",
    "dynamic_partition.time_zone" = "Asia/Shanghai"
)
""")
daily_window = lab.partition_window_summary(
    "m10_practice_window", title="The daily practice window"
)
assert len(daily_window) == 5
lab.expect_statement_error("""
INSERT INTO m10_practice_window VALUES (DATE_SUB(NOW(), INTERVAL 3 DAY), 1)
""", contains="no partition for this tuple",
     title="An event three days old has no Partition")
lab.execute("DROP TABLE m10_practice_window FORCE");

<a id="unit-u.b3b20b5e95384c9794b1381c424f4664"></a>

**Күтілетін нәтиже**

- Кестемен бірге жасалған **бес** Partition: бүгіннен екі күн бұрын, бүгін және екі күн кейін. Атаулары күнге сай, мысалы Asia/Shanghai бүгін 2026-10-22 болса, `p20261020`–`p20261024`.
- Әр Partition — бір жартылай ашық күн: біріншісі екі күн бұрынғы 00:00:00 бастап, бір күн бұрынғы 00:00:00 мәнін қоспай созылады.
- Үш күн бұрынғы енгізу `no partition for this tuple` қатесімен аяқталады: ол терезеден ескі және Dynamic Partitioning кеш оқиғаға Partition жасамайды.

`start` және `end` қазірден 24 сағаттық кезеңдерді емес, терезе уақыт белдеуіндегі бүгінге қатысты тұтас күндерді санайды, сондықтан терезе бес толық күнтізбелік күнді қамтиды: алдыңғы екі күн, бүгін және алдағы екі күн. Соңында кесте тек жаттығуға арналғандықтан `FORCE` арқылы жойылады; ол бастапқы күйге келтіру 4-бөлімдегідей қалпына келетін жою емес.

<a id="unit-u.2bbec6890a984a1daee61a973c53ec20"></a>

<a id="stop-the-doris-sandbox"></a>

### Doris сынақ ортасын тоқтату

Аяқтап, контейнер CPU мен жадын босатқыңыз келгенде осы қосымша ұяшықты орындаңыз. Ол Doris тоқтатады, бірақ контейнерді, образды, FE метадерек томын, BE сақтау томын және барлық `m10_*` нысандарын сақтайды. Сынақ ортасын пайдалануды жалғастырғыңыз келсе, өткізіп жіберіңіз.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris-rta
docker inspect --format 'container={{.State.Status}}' doris-rta
""", title="Stop the Doris sandbox");

<a id="unit-u.952a977fac88420682983e0da7d1e0ed"></a>

**Күтілетін нәтиже:** Docker `container=exited` деп хабарлайды. Кестелер атаулы томдарда қалады.

<a id="restart-the-doris-sandbox"></a>

### Doris сынақ ортасын қайта іске қосу

Бұл ұяшық идемпотентті: Docker Desktop пен контейнер тоқтаған, басталып жатқан немесе жұмыс істеп тұрған кезде орындала алады. Бар контейнерді бастайды не қайта пайдаланады, дұрыс күйді күтеді, қайта қосылып, сақталған сығу және қалпына келтіру кестелерін тексереді.

In [ ]:
lab.start_container()
lab.connect()
lab.execute("USE doris_course")

persisted = lab.sql("""
SELECT 'm10_events_none' AS object_name, COUNT(*) AS logical_rows
FROM m10_events_none
UNION ALL
SELECT 'm10_events_lz4f', COUNT(*) FROM m10_events_lz4f
UNION ALL
SELECT 'm10_events_zstd', COUNT(*) FROM m10_events_zstd
UNION ALL
SELECT 'm10_recovery_events', COUNT(*) FROM m10_recovery_events
ORDER BY object_name
""", title="Persisted Module 10 objects")
assert persisted.set_index("object_name")["logical_rows"].astype(int).to_dict() == {
    "m10_events_lz4f": 75_259,
    "m10_events_none": 75_259,
    "m10_events_zstd": 75_259,
    "m10_recovery_events": 3,
};

<a id="unit-u.3456dd02b0b94cc6b672cab68aede211"></a>

<a id="lab-complete"></a>

## Зертханалық жұмыс аяқталды

NO_COMPRESSION, LZ4F және ZSTD сол логикалық деректерді қалай сақтайтынын салыстырдыңыз, сығу нәтижелерін қайталанатын мәндер мен ортақ байт префикстерімен байланыстырдыңыз, сағаттық Dynamic Partition терезесінің толық мерзімі өткен Partition жоюын байқадыңыз, кеш оқиғаның терезеден қалай тыс қалатынын тексердіңіз және кездейсоқ жойылған Partition catalog recycle bin ішінен қалпына келтірдіңіз. Бұл нәтижелер бақыланатын зертханалық нысандарды сипаттайды; өндірістік жүктеме сынағын немесе тәуелсіз сақтық көшірмені алмастырмайды. Жоғарыдағы тоқтату және қайта бастау ұяшықтары `m10_*` кестелерін кейін тексеруге сақтайды.

Ресми анықтамалар: [Деректерді сығу](https://doris.apache.org/docs/4.x/table-design/column-compression/) · [SHOW DATA](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/data-and-status-management/SHOW-DATA/) · [Dynamic Partitioning](https://doris.apache.org/docs/4.x/table-design/data-partitioning/dynamic-partitioning/) · [SHOW CATALOG RECYCLE BIN](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/recycle/SHOW-CATALOG-RECYCLE-BIN/) · [RECOVER](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/recycle/RECOVER/)

<a id="unit-u.d7d21ce5beea470b8827bf14d3e11af9"></a>

<a id="next"></a>

## Келесі қадам

[10-тест: Apache Doris жүйесінде сақтау шығындары мен деректердің өмірлік циклін басқару](quiz10_data_lifecycle_and_recovery.ipynb) ашыңыз. Бұл — курстың соңғы зертханалық жұмысы: сабақ [курсты шолумен](course.md#where-the-course-leaves-you) және курс жасағанның бәрін тазалау жолымен аяқталады.